In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]



In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="embeddinggemma:300m")
sample_vector = embeddings.embed_query("!") # whatever text to get dimension
dimension = len(sample_vector)
print(f"Embedding dimension: {dimension}\n")
print("----")

# embed single documents at a time
for doc in documents:
    vector = embeddings.embed_query(doc.page_content)
    print(f"Document: {doc.page_content}\nEmbedding Vector: {vector[:10]}...\n")
    assert len(vector) == dimension

print("----")

# batch multiple documents: performance optimization
vectors = embeddings.embed_documents([doc.page_content for doc in documents])
for doc, vector in zip(documents, vectors):
    print(f"Document: {doc.page_content}\nEmbedding Vector: {vector[:10]}...\n")
    assert len(vector) == dimension

Embedding dimension: 768

----
Document: Dogs are great companions, known for their loyalty and friendliness.
Embedding Vector: [-0.09501003, -0.013341397, 0.03419346, -0.0052569658, -0.045418646, 0.010659981, -0.01419364, 0.011663996, 0.039012853, -0.016329143]...

Document: Cats are independent pets that often enjoy their own space.
Embedding Vector: [-0.09489994, 0.054017987, -0.036460962, 0.024761423, -0.026513116, 0.00080759806, -0.03563051, 0.03358379, 0.02463776, -0.03692745]...

----
Document: Dogs are great companions, known for their loyalty and friendliness.
Embedding Vector: [-0.09501003, -0.013341397, 0.03419346, -0.0052569658, -0.045418646, 0.010659981, -0.01419364, 0.011663996, 0.039012853, -0.016329143]...

Document: Cats are independent pets that often enjoy their own space.
Embedding Vector: [-0.09489994, 0.054017987, -0.036460962, 0.024761423, -0.026513116, 0.00080759806, -0.03563051, 0.03358379, 0.02463776, -0.03692745]...



In [ ]:
import numpy as np

# the angle between two vectors (ranging from -1 (opposite) to 1 (same direction))
# => Are these vectors pointing in the same direction? (no matters their lenght)
def cosine_similarity(vec1, vec2):
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2)) #  dot product / (magnitude1 * magnitude2)

# the straight-line distance between two vectors
def euclidean_distance(vec1, vec2):
    return np.linalg.norm(np.array(vec1) - np.array(vec2))

# measures both the angle and the magnitudes (lengths) of the vectors.
# => Are these vectors pointing in the same direction and how long are they?
# since embedding is typically normalized (thinking in 2D are like arrows from the origin ot points on a unit circle), dot product is equivalent to cosine similarity and simpler to compute
def dot_product(vec1, vec2):
    return np.dot(vec1, vec2)

query_embedding = embeddings.embed_query("I love my dog!")
for i, doc in enumerate(documents):
    similarity = cosine_similarity(query_embedding, vectors[i])
    distance = euclidean_distance(query_embedding, vectors[i])
    product = dot_product(query_embedding, vectors[i])
    
    print(f"Document '{doc.page_content}':")    
    print(f"  Euclidean Distance: {distance}")
    print(f"  Cosine Similarity: {similarity}")
    print(f"  Dot Product: {product}\n")

magnitude vec1: 0.9999997070538563
magnitude vec2: 1.0000000955429829
Document 'Dogs are great companions, known for their loyalty and friendliness.':
  Euclidean Distance: 0.9816634623244234
  Cosine Similarity: 0.5181683282535797
  Dot Product: 0.5181682259654994

magnitude vec1: 0.9999997070538563
magnitude vec2: 0.9999999515358998
Document 'Cats are independent pets that often enjoy their own space.':
  Euclidean Distance: 1.1162127761573981
  Cosine Similarity: 0.37703430648466674
  Dot Product: 0.37703417776129755



In [25]:
import numpy as np

print("=" * 80)
print("WHY COSINE SIMILARITY ≈ DOT PRODUCT FOR NORMALIZED EMBEDDINGS")
print("=" * 80)

# ============================================================================
# 1. MATHEMATICAL RELATIONSHIP
# ============================================================================
print("\n📐 MATHEMATICAL RELATIONSHIP\n")

explanation = """
Cosine Similarity = dot(A, B) / (||A|| × ||B||))
Dot Product       = dot(A, B)

If vectors are NORMALIZED (scaled length to 1) (||A|| = ||B|| = 1), then:
    Cosine Similarity = dot(A, B) / (1 × 1) = dot(A, B)
    
Therefore: Cosine Similarity ≡ Dot Product for unit vectors
"""
print(explanation)

# ============================================================================
# 2. WHY EMBEDDING MODELS NORMALIZE
# ============================================================================
print("\n\n🤔 WHY DO EMBEDDING MODELS NORMALIZE?\n")

reasons = """
Modern embedding models typically output NORMALIZED embeddings because:

1. **Efficiency**: Dot product is faster than cosine similarity
   - No need to compute norms during search
   - Can use optimized linear algebra libraries

2. **Consistency**: Makes magnitude irrelevant
   - Only direction matters for similarity
   - Prevents length bias

3. **Mathematical Properties**:
   - Dot product of unit vectors ∈ [-1, 1] (same range as cosine)
   - Easy to convert to distances: distance = 1 - similarity
   - Works well with approximate nearest neighbor algorithms

4. **Vector Database Optimization**:
   - Most vector DBs assume normalized vectors
   - Enables faster approximate search
"""
print(reasons)

# ============================================================================
# 3. PRACTICAL IMPLICATIONS
# ============================================================================
print("\n\n💼 PRACTICAL IMPLICATIONS\n")

implications = """

✓ Use DOT PRODUCT for similarity search (faster, same results)
  → Most vector databases default to this for normalized embeddings

✓ Use COSINE SIMILARITY if:
  → You're mixing embeddings from different sources
  → You're not sure if vectors are normalized
  → You need robustness against scale differences

✓ Use EUCLIDEAN DISTANCE when:
  → You care about absolute differences (not just direction)
  → Working with non-normalized vectors
  → Note: For normalized vectors, euclidean ≈ 2×(1 - cosine)
"""
print(implications)

WHY COSINE SIMILARITY ≈ DOT PRODUCT FOR NORMALIZED EMBEDDINGS

📐 MATHEMATICAL RELATIONSHIP


Cosine Similarity = dot(A, B) / (||A|| × ||B||))
Dot Product       = dot(A, B)

If vectors are NORMALIZED (scaled length to 1) (||A|| = ||B|| = 1), then:
    Cosine Similarity = dot(A, B) / (1 × 1) = dot(A, B)

Therefore: Cosine Similarity ≡ Dot Product for unit vectors



🤔 WHY DO EMBEDDING MODELS NORMALIZE?


Modern embedding models typically output NORMALIZED embeddings because:

1. **Efficiency**: Dot product is faster than cosine similarity
   - No need to compute norms during search
   - Can use optimized linear algebra libraries

2. **Consistency**: Makes magnitude irrelevant
   - Only direction matters for similarity
   - Prevents length bias

3. **Mathematical Properties**:
   - Dot product of unit vectors ∈ [-1, 1] (same range as cosine)
   - Easy to convert to distances: distance = 1 - similarity
   - Works well with approximate nearest neighbor algorithms

4. **Vector Database Optim

In [ ]:
!uv pip install -U "langchain-openai"

In [16]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") # text-embedding-3-large
sample_vector = embeddings.embed_query("!") # whatever text to get dimension
dimension = len(sample_vector)
print(f"Embedding dimension: {dimension}\n")
print("----")

# batch multiple documents: performance optimization
_ = embeddings.embed_documents([doc.page_content for doc in documents])
for doc, vector in zip(documents, _):
    print(f"Document: {doc.page_content}\nEmbedding Vector: {vector[:10]}...\n")
    assert len(vector) == dimension

Embedding dimension: 1536

----
Document: Dogs are great companions, known for their loyalty and friendliness.
Embedding Vector: [0.02680193819105625, -0.011360367760062218, 0.0001673534861765802, 0.05169525370001793, 0.024223268032073975, 0.011604021303355694, -0.02613189071416855, 0.040933869779109955, -0.01827405020594597, 0.01359386183321476]...

Document: Cats are independent pets that often enjoy their own space.
Embedding Vector: [0.039093609899282455, -0.015071647241711617, 0.03899605944752693, 0.03431360423564911, 0.04472718760371208, 0.024717014282941818, -0.0024037205148488283, -0.010822076350450516, 0.030826151371002197, 0.03263084962964058]...



In [ ]:
# manage large documents and batch size based on token counts

from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import tiktoken

# expand each document size to simulate larger embeddings
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness." * 100_000,
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space." * 100_000,
        metadata={"source": "mammal-pets-doc"},
    ),
]

MAX_TOKENS_PER_BATCH = 300_000 * 0.8
MAX_CHUNK_SIZE = 10_000

def chunk(documents: list[Document]) -> list[Document]:
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=MAX_CHUNK_SIZE, 
        chunk_overlap=int(MAX_CHUNK_SIZE * 0.1),
        length_function=len,  # Character count
    )
    chunked_documents = []
    for doc in documents:
        if len(doc.page_content) <= MAX_CHUNK_SIZE:
            chunked_documents.append(doc)
            continue
        # Split the text
        chunks = text_splitter.split_text(doc.page_content)
        print(f"  Split document into {len(chunks)} chunks")
        for chunk_text in chunks:
            chunked_documents.append(
                Document(page_content=chunk_text, metadata=doc.metadata)
            )
    return chunked_documents

def _count_tokens(text: str) -> int:
    """Count tokens in text using tiktoken or fallback estimation"""
    try:
        encoding = tiktoken.get_encoding("cl100k_base")
        tokens = encoding.encode(text)
        count = len(tokens)
        return count
    except Exception as e:
        # fallback: rough estimation (1 token ≈ 4 characters)
        estimate = len(text) // 4
        print(f"Fallback estimation: {estimate:,} tokens (error: {e})")
        return estimate

def _batch_documents_by_tokens(documents: list[Document], max_tokens: int = MAX_TOKENS_PER_BATCH) -> list[list[Document]]:
    """Split documents into batches based on token count
    
    If a single document exceeds max_tokens, it will be placed in its own batch
    and should be split further by the caller.
    """
    if not documents:
        return []
    
    batches = []
    current_batch = []
    current_token_count = 0
    
    for doc in documents:
        doc_tokens = _count_tokens(doc.page_content)
        
        # If single document exceeds limit, warn and put it in its own batch
        if doc_tokens > max_tokens:
            # Save current batch if not empty
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_token_count = 0
            # Add oversized document as its own batch
            batches.append([doc])
            continue
        
        # Check if adding this document exceeds the limit
        if current_token_count + doc_tokens > max_tokens:
            # Start new batch if current batch is not empty
            if current_batch:
                batches.append(current_batch)
            # Reset current batch
            current_batch = [doc]
            current_token_count = doc_tokens
        else:
            # Add to current batch
            current_batch.append(doc)
            current_token_count += doc_tokens
    
    # Add final batch if not empty
    if current_batch:
        batches.append(current_batch)
    
    return batches

# Test batching
print(f"Max tokens per batch: {MAX_TOKENS_PER_BATCH:,}\n")
chunked_docs = chunk(documents)
batches = _batch_documents_by_tokens(chunked_docs)
print(f"\n{'='*80}")
print(f"RESULT: {len(chunked_docs)} documents → {len(batches)} batches")
print(f"{'='*80}")

for i, batch in enumerate(batches):
    total_tokens = sum(_count_tokens(doc.page_content) for doc in batch)
    print(f"Batch {i+1}: {len(batch)} documents, ~{total_tokens:,} tokens")

if False:
    for batch in batches:
        _ = embeddings.embed_documents([doc.page_content for doc in batch])
        for doc, vector in zip(batch, _):
            print(f"Document: {doc.page_content[:10]}\nEmbedding Vector: {vector[:10]}...\n")